# Lesson 05: Decision Trees, Random Forests, and Ensembles - Full Solution Notebook

Course: Machine Learning for Chemical Engineering

Focus: learned thresholds, tree visualization, feature hierarchy, max_depth, overfitting, Random Forests, and regression/classification.

## Learning goals
1. Build a synthetic but physically intuitive cooling-system dataset.
2. Train a shallow decision tree and visualize its learned hierarchy.
3. Compare shallow trees, deep trees, and Random Forests.
4. Use `max_depth` as a regularization hyperparameter.
5. Use trees for both regression and classification.

## Colab use
Run cells from top to bottom. The notebook is self-contained and uses synthetic data so it can run from GitHub/Colab without external files.

This solution is one possible implementation. Use it only after attempting the exercise notebook.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, accuracy_score, ConfusionMatrixDisplay

rng = np.random.default_rng(5)
n = 260

coolant_flow = rng.uniform(0.7, 5.5, n)
heat_load = rng.uniform(35, 95, n)
ambient_temp = rng.uniform(20, 38, n)
vibration = rng.gamma(shape=2.0, scale=0.35, size=n)

thermal_resistance = 1.25 / np.maximum(coolant_flow, 0.35)
low_flow_penalty = np.where(coolant_flow < 2.7, 12.0 + 4.0 * (2.7 - coolant_flow), 0.0)
T_max = (
    42
    + thermal_resistance * heat_load
    + 0.85 * (ambient_temp - 25)
    + 2.2 * vibration
    + low_flow_penalty
    + rng.normal(0, 2.6, n)
)

overheat = (T_max > 85).astype(int)

X = pd.DataFrame({
    "coolant_flow_L_min": coolant_flow,
    "heat_load_W": heat_load,
    "ambient_temp_C": ambient_temp,
    "vibration_index": vibration,
})
y_reg = T_max
y_cls = overheat

X_train, X_test, y_train, y_test, yc_train, yc_test = train_test_split(
    X, y_reg, y_cls, test_size=0.25, random_state=2, stratify=y_cls
)

models = {
    "shallow_tree_depth_3": DecisionTreeRegressor(max_depth=3, min_samples_leaf=8, random_state=0),
    "deep_tree_unlimited": DecisionTreeRegressor(max_depth=None, min_samples_leaf=1, random_state=0),
    "random_forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=7,
        min_samples_leaf=3,
        max_features="sqrt",
        random_state=0,
    ),
}

print("Regression test RMSE")
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, pred, squared=False)
    print(f"{name:24s}: {rmse:6.2f} deg C")

fig, ax = plt.subplots(figsize=(12, 5))
plot_tree(
    models["shallow_tree_depth_3"],
    feature_names=X.columns,
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax,
)
ax.set_title("Shallow regression tree: learned feature hierarchy")
plt.show()

importance = pd.DataFrame({
    "feature": X.columns,
    "shallow_tree": models["shallow_tree_depth_3"].feature_importances_,
    "random_forest": models["random_forest"].feature_importances_,
}).sort_values("random_forest", ascending=False)
print(importance.to_string(index=False))

flow_grid = np.linspace(0.7, 5.5, 220)
scenario = pd.DataFrame({
    "coolant_flow_L_min": flow_grid,
    "heat_load_W": 72.0,
    "ambient_temp_C": 30.0,
    "vibration_index": 0.8,
})

plt.figure(figsize=(8, 4.6))
for name, model in models.items():
    plt.plot(flow_grid, model.predict(scenario), label=name)
plt.axvline(2.7, color="black", linestyle="--", linewidth=1, label="true low-flow threshold")
plt.xlabel("Coolant flow rate (L/min)")
plt.ylabel("Predicted maximum temperature (deg C)")
plt.title("Effect of max_depth and Random Forest averaging")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

classifier = DecisionTreeClassifier(max_depth=3, min_samples_leaf=8, random_state=0)
classifier.fit(X_train, yc_train)
yc_pred = classifier.predict(X_test)
print("Classification accuracy:", round(accuracy_score(yc_test, yc_pred), 3))

ConfusionMatrixDisplay.from_predictions(
    yc_test,
    yc_pred,
    display_labels=["safe", "overheat"],
    cmap="Blues",
)
plt.title("Decision-tree classifier: thermal safety")
plt.show()


## Interpretation

The shallow tree is a learned model because the algorithm selected the feature thresholds from data. In most runs, coolant flow appears near the top of the tree because it controls the heat-removal bottleneck. That is the useful feature hierarchy we get almost for free.

A very shallow tree is readable but may underfit. An unlimited tree can memorize noisy measurements and create unstable thresholds. Random Forests reduce that instability by averaging many decorrelated trees, which is why they are such a strong default model for tabular engineering data.

The same structure can model reactor cooling, battery thermal safety, electronics cooling, pump cavitation risk, material failure, HVAC performance, or any system with threshold-like operating regimes.
